# vLLM으로 개인 프로젝트 모델 서빙 실습 (12주차)

**과제**
1. 개인 프로젝트(Science_Chatbot)에 쓰이는 공개 가중치 모델을 vLLM으로 서빙해본다.
2. (선택, 이번엔 생략) 서빙 엔진을 Docker + EC2로 배포.

**모델**: `Qwen/Qwen2.5-1.5B-Instruct` — Science_Chatbot의 `Qwen-tuned`가 QLoRA로 파인튜닝한 베이스와 같은 계열.

**핵심 포인트**: 지금 Science_Chatbot은 이 모델을 llama-server(GGUF)로 서빙 중인데, `models.py`는 `ChatOpenAI(base_url=...)`로만 연결해서 서빙 엔진이 무엇인지 모른다(OpenAI 호환 API 뒤에 격리돼 있음). vLLM도 OpenAI 호환 서버를 제공하므로, **프로젝트 코드는 한 줄도 안 고치고 `base_url`만 바꾸면** 서빙 엔진을 통째로 교체할 수 있다는 걸 이 노트북에서 확인한다.

**Colab 런타임**: 상단 메뉴 `런타임 > 런타임 유형 변경`에서 GPU(T4 이상) 선택 후 진행.

In [ ]:
!nvidia-smi

## 1. vLLM 설치

In [ ]:
!pip install -q vllm

## 2. vLLM OpenAI 호환 서버 실행

`vllm serve`는 llama-server와 마찬가지로 OpenAI 호환 REST API(`/v1/chat/completions`)를 연다. Colab 노트북 안에서는 서버가 셀을 점유하며 블로킹되므로, 백그라운드 프로세스로 띄우고 `/health`로 준비 여부를 폴링한다.

In [ ]:
import subprocess, time, requests

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000

server_proc = subprocess.Popen(
    ["vllm", "serve", MODEL_ID, "--port", str(PORT), "--max-model-len", "4096"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

# 모델 로딩 포함 몇 분 걸릴 수 있음
for _ in range(180):
    try:
        if requests.get(f"http://localhost:{PORT}/health").status_code == 200:
            print("vLLM 서버 준비 완료")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(2)
else:
    raise RuntimeError("서버가 제한 시간 안에 안 떴음 — 아래 로그 확인: server_proc.stdout")

## 3. 서빙 확인 — OpenAI 호환 API 직접 호출

In [ ]:
resp = requests.post(
    f"http://localhost:{PORT}/v1/chat/completions",
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": "만유인력의 법칙을 한 문장으로 설명해줘."}],
        "max_tokens": 200,
    },
)
print(resp.json()["choices"][0]["message"]["content"])

## 4. Science_Chatbot 통합 코드로 호출 (드롭인 교체 확인)

`models.py`의 `Qwen-tuned` 항목이 실제로 쓰는 것과 똑같은 `ChatOpenAI` 클래스로 이 vLLM 서버를 불러본다 — `base_url`만 바뀔 뿐 나머지 코드는 llama-server용과 동일하다.

In [ ]:
!pip install -q langchain-openai

from langchain_openai import ChatOpenAI

# Science_Chatbot의 models.py와 동일한 구성 — base_url만 이 Colab 서버로 바뀜
llm = ChatOpenAI(
    base_url=f"http://localhost:{PORT}/v1",
    api_key="not-needed",  # 로컬 서버는 키 검사 안 함(필드가 필수라 더미값)
    model=MODEL_ID,
)

response = llm.invoke("전자기 유도가 뭐야?")
print(response.content)

## 5. 서버 종료

In [ ]:
server_proc.terminate()
server_proc.wait()
print("서버 종료됨")